In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import gc
import random
import warnings
import ctypes

In [2]:
def set_seed(seed: int) -> int:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    
    warnings.filterwarnings("ignore")
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)

    def seed_worker(worker_id: int) -> None:
        worker_seed = torch.initial_seed() % (2 ** 32)
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    
    print(f"Random seed: {seed}")
    return seed_worker

def clear_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except:
        pass

In [3]:
DATASET_CONFIG = {
    "sunspots": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/sunspots_dataset.csv",
        "time_col": "Date",
        "target_col": "Monthly Mean Total Sunspot Number",
        "type": "univariate"
    },
    "appliances_energy": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/appliances_energy_dataset.csv",
        "time_col": "date",
        "target_col": None,  
        "type": "multivariate"
    },
    "beijing_air_quality": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/beijing_air_quality_dataset.csv",
        "time_col": ["year", "month", "day", "hour"], 
        "target_col": None,  
        "type": "multivariate"
    },
    "hanoi_air_quality": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/hanoi_air_quality_dataset.csv",
        "time_col": "Local Time",
        "target_col": None,  
        "type": "multivariate"
    },
    "bitcoin": {
        "path": "/kaggle/input/datasets/minh3m25/time-series-dataset/bitcoin_dataset.csv",
        "time_col": "Timestamp",
        "target_col": "Open",  
        "type": "univariate"
    }   
}

In [4]:
class TSWindowDataset(Dataset):
    def __init__(self, data_df, seq_len, pred_len):
        self.data = torch.tensor(data_df.values, dtype=torch.float32)
        self.seq_len = seq_len
        self.pred_len = pred_len
        
    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1
        
    def __getitem__(self, idx):
        X = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return X, y

class TimeSeriesDataset:
    def __init__(self, name, seq_len=24, pred_len=1, train_ratio=0.7, val_ratio=0.2, 
                 batch_size=64, num_workers=0, worker_init_fn=None, generator=None):
        """
        :param num_workers: Số luồng tải dữ liệu
        :param worker_init_fn: Hàm khởi tạo seed cho từng worker
        :param generator: Bộ sinh số ngẫu nhiên của PyTorch
        """
        if name not in DATASET_CONFIG:
            raise ValueError(f"Tên dataset '{name}' không tồn tại trong DATASET_CONFIG.")
        
        self.name = name
        self.config = DATASET_CONFIG[name].copy() 
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.batch_size = batch_size
        
        # Thêm các biến để cấu hình DataLoader tái lập
        self.num_workers = num_workers
        self.worker_init_fn = worker_init_fn
        self.generator = generator
        
        self.df_train = None
        self.df_val = None
        self.df_test = None
        
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        
        self._load_and_split_data()
        
    def _load_and_split_data(self):
        df = pd.read_csv(self.config['path'])
        time_col = self.config['time_col']
        
        if isinstance(time_col, list):
            df['datetime_merged'] = pd.to_datetime(df[time_col])
            df.drop(columns=time_col, inplace=True)
            time_col = 'datetime_merged'
        
        if time_col and time_col in df.columns:
            df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
            df = df.sort_values(by=time_col).reset_index(drop=True)
            df.set_index(time_col, inplace=True)

        if 'No' in df.columns:
            df.drop(columns=['No'], inplace=True)
            
        df = df.select_dtypes(include=[np.number]) # Chỉ giữ lại dữ liệu dạng số
            
        if self.config['type'] == 'univariate':
            df = df[[self.config['target_col']]]
        elif self.config['type'] == 'multivariate':
            self.config['target_col'] = list(df.columns)
            
        df = df.dropna()
            
        n = len(df)
        train_end = int(n * self.train_ratio)
        val_end = int(n * (self.train_ratio + self.val_ratio))
        
        self.df_train = df.iloc[:train_end]
        self.df_val = df.iloc[train_end:val_end]
        self.df_test = df.iloc[val_end:]
        
        self.train_dataset = TSWindowDataset(self.df_train, self.seq_len, self.pred_len)
        self.val_dataset = TSWindowDataset(self.df_val, self.seq_len, self.pred_len)
        self.test_dataset = TSWindowDataset(self.df_test, self.seq_len, self.pred_len)
        
    def get_loaders(self):
        # Đưa worker_init_fn và generator vào DataLoader
        train_loader = DataLoader(
            self.train_dataset, 
            batch_size=self.batch_size, 
            shuffle=True,
            num_workers=self.num_workers,
            worker_init_fn=self.worker_init_fn,
            generator=self.generator
        )
        
        val_loader = DataLoader(
            self.val_dataset, 
            batch_size=self.batch_size, 
            shuffle=False,
            num_workers=self.num_workers,
            worker_init_fn=self.worker_init_fn,
            generator=self.generator
        )
        
        test_loader = DataLoader(
            self.test_dataset, 
            batch_size=self.batch_size, 
            shuffle=False,
            num_workers=self.num_workers,
            worker_init_fn=self.worker_init_fn,
            generator=self.generator
        )
        return train_loader, val_loader, test_loader
        
    def print_info(self):
        X_tr, y_tr = self.train_dataset[0]
        
        train_batches = len(self.train_dataset) // self.batch_size + (1 if len(self.train_dataset) % self.batch_size != 0 else 0)
        val_batches = len(self.val_dataset) // self.batch_size + (1 if len(self.val_dataset) % self.batch_size != 0 else 0)
        test_batches = len(self.test_dataset) // self.batch_size + (1 if len(self.test_dataset) % self.batch_size != 0 else 0)
        
        print(f"=== THÔNG TIN DATASET: {self.name.upper()} ===")
        print(f"- Phân loại: {self.config['type'].upper()}")
        
        if self.config['type'] == 'multivariate':
            print(f"- Số lượng đặc trưng (Features): {len(self.config['target_col'])} cột")
        else:
            print(f"- Biến mục tiêu (Target): {self.config['target_col']}")
            
        print(f"- Kích thước chuỗi (seq_len, pred_len): ({self.seq_len}, {self.pred_len})")
        print(f"- Batch size: {self.batch_size} | Num workers: {self.num_workers}")
        print("-" * 50)
        print("KÍCH THƯỚC DỮ LIỆU HUẤN LUYỆN (TENSORS & DATALOADER):")
        
        print(f" 1. Tập Train ({self.train_ratio*100:.0f}%):")
        print(f"    + Số cửa sổ: {len(self.train_dataset)} -> Chia thành {train_batches} batches")
        print(f"    + Shape X 1 mẫu : ({X_tr.shape[0]}, {X_tr.shape[1]})")
        
        print(f" 2. Tập Val ({self.val_ratio*100:.0f}%):")
        print(f"    + Số cửa sổ: {len(self.val_dataset)} -> Chia thành {val_batches} batches")
        
        print(f" 3. Tập Test ({(1 - self.train_ratio - self.val_ratio)*100:.0f}%):")
        print(f"    + Số cửa sổ: {len(self.test_dataset)} -> Chia thành {test_batches} batches")
        print("==================================================")

In [5]:
RANDOM_SEED = 42

SEED_WORKER = set_seed(RANDOM_SEED)

g = torch.Generator()
g.manual_seed(RANDOM_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

Random seed: 42
Device: cuda


In [6]:
dataset = TimeSeriesDataset(
    name="beijing_air_quality", 
    seq_len=24, 
    pred_len=1, 
    batch_size=32,
    num_workers=2,                
    worker_init_fn=SEED_WORKER,
    generator=g                   
)

dataset.print_info()

train_loader, val_loader, test_loader = dataset.get_loaders()

=== THÔNG TIN DATASET: BEIJING_AIR_QUALITY ===
- Phân loại: MULTIVARIATE
- Số lượng đặc trưng (Features): 11 cột
- Kích thước chuỗi (seq_len, pred_len): (24, 1)
- Batch size: 32 | Num workers: 2
--------------------------------------------------
KÍCH THƯỚC DỮ LIỆU HUẤN LUYỆN (TENSORS & DATALOADER):
 1. Tập Train (70%):
    + Số cửa sổ: 22289 -> Chia thành 697 batches
    + Shape X 1 mẫu : (24, 11)
 2. Tập Val (20%):
    + Số cửa sổ: 6351 -> Chia thành 199 batches
 3. Tập Test (10%):
    + Số cửa sổ: 3164 -> Chia thành 99 batches
